### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
import matplotlib.ticker as mtick

### Configure Settings

In [ ]:
pd.set_option('display.max_columns', None)
%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

### Load datasets

In [ ]:
raw_customers = pd.read_csv('../data/customer_signups.csv')
customers = pd.read_csv('../data/customer_signups.csv')
tickets = pd.read_csv('../data/support_tickets.csv')

print(f"Signups: {customers.shape}")
print(f"Tickets: {tickets.shape}")

### Data Exploration

In [ ]:
customers.head()
customers.info()
customers.describe(include='all')

In [ ]:
tickets.head()
tickets.info()
tickets.describe(include='all')

### Data Cleaning

#### Convert signup_date to datetime

In [ ]:
invalid_dates = customers[
    pd.to_datetime(customers['signup_date'], errors='coerce').isna()
]
print("Invalid signup_date entries:")
display(invalid_dates[['signup_date']])


customers['signup_date'] = customers['signup_date'].replace('not a date', np.nan)
customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')


print(f"Valid dates: {customers['signup_date'].notna().sum()}")
print(f"Invalid dates: {customers['signup_date'].isna().sum()}")

display(customers.loc[0, 'signup_date'])

#### Standardize inconsistent text values & Handle missing values

In [ ]:
cat_cols = customers.select_dtypes(include=['object','datetime64','bool']).columns
cat_cols

for col in cat_cols:
    print(f"\n--- {col} ---")
    print(customers[col].value_counts(dropna=False))

##### customer_id

In [ ]:
customers['customer_id'] = customers['customer_id'].astype(str).str.strip()

customers['customer_id'] = customers['customer_id'].replace({'nan': np.nan})

##### name

In [ ]:
customers['name'] = (
    customers['name']
    .astype(str)
    .str.replace(r'^(Mr\.|Mrs\.|Ms\.|Miss)\s+', '', regex=True)
    .str.title()
    .replace({'Nan': np.nan})
)

##### email

In [ ]:
customers['email'] = (
    customers['email']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({'nan': np.nan})
)

customers['email_missing'] = customers['email'].isna()


##### source

In [ ]:
source_map = {
    'Youtube': 'YouTube',
    'Linkedin': 'LinkedIn',
    '??': 'Unknown',
    np.nan: 'Unknown',
    'Nan': np.nan
}

customers['source'] = (
    customers['source']
    .astype(str)
    .str.strip()
    .str.title()
    .replace(source_map)
)

##### region

In [ ]:
customers['region'] = (
    customers['region']
    .astype(str)
    .str.strip()
    .str.title()
    .replace({'Nan': np.nan})
)

customers['region'] = customers['region'].fillna('Unknown')
customers['region_missing'] = customers['region'] == 'Unknown'

##### plan_selected

In [ ]:
plan_map = {
    'basic': 'Basic',
    'Basic': 'Basic',
    'pro': 'Pro',
    'Pro': 'Pro',
    'premium': 'Premium',
    'Premium': 'Premium',
    'PREMIUM': 'Premium',
    'PRO': 'Pro',
    'prem': 'Premium',
    'Unknownplan': 'Unknown'
}

customers['plan_selected'] = (
    customers['plan_selected']
    .astype(str)
    .str.strip()
    .replace(plan_map)
    .replace({'nan': np.nan})
)

##### marketing_opt_in

In [ ]:
customers['marketing_opt_in'] = (
    customers['marketing_opt_in']
    .astype(str)
    .str.strip()
    .str.title()
    .replace({'Nil': np.nan, 'Nan': np.nan})
)

##### age

In [ ]:
customers['age'] = customers['age'].replace({'unknown': np.nan, 'thirty': '30'})

def clean_age(x):
    try:
        age = int(float(x))  
        if age < 16 or age > 100:
            return f"UNUSUAL_{age}"
        return age
    except:
        return np.nan

customers['age_cleaned'] = customers['age'].apply(clean_age)


def age_source(x):
    if pd.isna(x):
        return 'Missing'
    elif isinstance(x, str) and x.startswith('UNUSUAL'):
        return 'Unusual'
    else:
        return 'Original'

customers['age_source'] = customers['age_cleaned'].apply(age_source)


median_age = customers.loc[customers['age_source'] == 'Original', 'age_cleaned'].median()
customers.loc[customers['age_source'] == 'Missing', 'age_cleaned'] = median_age
customers.loc[customers['age_source'] == 'Missing', 'age_source'] = 'Imputed'


def finalize_age(x):
    if isinstance(x, str) and x.startswith('UNUSUAL'):
        return x
    else:
        return int(x)

customers['age_cleaned'] = customers['age_cleaned'].apply(finalize_age)

unusual_ages = customers[customers['age_source'] == 'Unusual']
print("Unusual ages detected:")
display(unusual_ages[['age', 'age_cleaned', 'age_source']])


##### gender

In [ ]:
gender_map = {
    'Male': 'Male',
    'male': 'Male',
    'Female': 'Female',
    'FEMALE': 'Female',
    'Non-Binary': 'Non-Binary',
    'Other': 'Other',
    '123': np.nan
}

customers['gender'] = (
    customers['gender']
    .astype(str)
    .str.strip()
    .replace(gender_map)
    .replace({'nan': np.nan})
)

#### Remove Duplicates

In [ ]:
initial_count = len(customers)


non_null_ids = customers[customers['customer_id'].notna()].copy()
null_ids = customers[customers['customer_id'].isna()].copy()


duplicate_rows = non_null_ids[non_null_ids.duplicated(subset='customer_id', keep=False)]
duplicate_ids = duplicate_rows['customer_id'].unique()

print(f"Found {len(duplicate_ids)} duplicated customer_id(s): {duplicate_ids}")
display(duplicate_rows.sort_values('customer_id'))


non_null_ids = non_null_ids.drop_duplicates(subset='customer_id', keep='first')


if not null_ids.empty:
    
    existing_nums = (
        non_null_ids['customer_id']
        .str.extract(r'(\d+)$')[0]
        .astype(int)
    )

    start_num = existing_nums.max() + 1
    end_num = start_num + len(null_ids)

    new_ids = [f"CUST{num:05d}" for num in range(start_num, end_num)]
    null_ids['customer_id'] = new_ids

    print(f"Assigned new IDs to {len(null_ids)} missing entries, starting from {new_ids}")


customers = (
    pd.concat([non_null_ids, null_ids])
    .sort_index()
    .reset_index(drop=True)
)


duplicates_removed = initial_count - len(customers)
print(f"Removed {duplicates_removed} duplicate customer_id(s)")
print(f"Total customers after cleanup: {len(customers)}")

### Data Quality Summary

In [ ]:
def combined_missing_summary(raw_df, clean_df):
    common_cols = [col for col in clean_df.columns if col in raw_df.columns]
    raw_missing = raw_df[common_cols].isna().sum()
    clean_missing = clean_df[common_cols].isna().sum()

    missing_df = pd.DataFrame({
        'Missing (Raw)': raw_missing,
        '% Missing (Raw)': (raw_missing / len(raw_df) * 100).round(2),
        'Missing (Cleaned)': clean_missing,
        '% Missing (Cleaned)': (clean_missing / len(clean_df) * 100).round(2),
    })

    missing_df['Change (Count)'] = missing_df['Missing (Raw)'] - missing_df['Missing (Cleaned)']
    missing_df['Change (%)'] = (
        missing_df['% Missing (Raw)'] - missing_df['% Missing (Cleaned)']
    ).round(2)

    missing_df = missing_df.sort_values('% Missing (Raw)', ascending=False)

    missing_df['Trend'] = np.where(
    missing_df['Change (Count)'] > 0, '🟢 Decreased',
    np.where(missing_df['Change (Count)'] < 0, '🔴 Increased', '⚪ No Change')
)


    print("\nCombined Missing Values Summary (Before vs After Cleaning):")
    display(missing_df)

    return missing_df


def category_comparison(raw_df, clean_df, cat_columns):
    print("\nCategory Value Comparison (Raw vs Cleaned):")
    comparison = {}

    for col in cat_columns:
        raw_vals = set(raw_df[col].dropna().astype(str).unique())
        clean_vals = set(clean_df[col].dropna().astype(str).unique())

        changed = raw_vals.symmetric_difference(clean_vals)
        mapping = {
            'Original (Raw)': sorted(list(raw_vals)),
            'After Cleaning': sorted(list(clean_vals)),
            'Changed Values': sorted(list(changed)) if changed else ['No Change']
        }

        comparison[col] = mapping

        print(f"\nColumn: {col}")
        print(f"Original (Raw): {mapping['Original (Raw)']}")
        print(f"After Cleaning: {mapping['After Cleaning']}")
        print(f"Changed Values: {mapping['Changed Values']}")

    return comparison


cat_columns = ['plan_selected','marketing_opt_in','gender','source','region']

combined_missing = combined_missing_summary(raw_customers, customers)
category_changes = category_comparison(raw_customers, customers, cat_columns)

numeric_ages = customers.loc[customers['age_source'].isin(['Original','Imputed']), 'age_cleaned']
unusual_ages = customers.loc[customers['age_source']=='Unusual','age_cleaned']

print("\nNumeric Age Summary (Cleaned):")
display(numeric_ages.value_counts().sort_index())

print("\nUnusual Ages Detected (Cleaned):")
display(unusual_ages.value_counts())


### Summary Outputs

#### Sign-ups per Week

In [ ]:
signups_per_week = (
    customers
    .set_index('signup_date')
    .resample('W')
    .size()
    .reset_index(name='Signups')
)

print("Sign-ups Per Week:")
display(signups_per_week)

plt.figure(figsize=(12,5))
plt.bar(signups_per_week['signup_date'], signups_per_week['Signups'], 
        width=5, color='skyblue', alpha=0.7, label='Weekly Sign-ups')
sns.lineplot(data=signups_per_week, x='signup_date', y='Signups', color='red', marker='o', label='Trend')

for i in range(0, len(signups_per_week), 2):
    plt.axvspan(signups_per_week['signup_date'][i], 
                signups_per_week['signup_date'][i]+pd.Timedelta(days=6), 
                color='gray', alpha=0.1)

plt.gca().xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO, interval=2))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

plt.title("Weekly Sign-ups Trend")
plt.xlabel("Week")
plt.ylabel("Number of Sign-ups")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


#### Sign-ups by source, region, and plan_selected

In [ ]:
signups_by_factors = (
    customers.groupby(['source', 'region', 'plan_selected'])
    .size()
    .reset_index(name='Signups')
    .sort_values('Signups', ascending=False)
)

print("Sign-ups by Source, Region, and Plan:")
display(signups_by_factors.head(10))

top_sources = customers['source'].value_counts().reset_index()
top_sources.columns = ['Source', 'Signups']


region_counts = customers['region'].value_counts().reset_index()
region_counts.columns = ['Region', 'Signups']


plan_counts = customers['plan_selected'].value_counts().reset_index()
plan_counts.columns = ['Plan', 'Signups']

fig, axes = plt.subplots(1, 3, figsize=(18,5))

sns.barplot(
    data=top_sources, 
    x='Signups', 
    y='Source', 
    hue='Source', 
    dodge=False, 
    palette='pastel', 
    legend=False,
    ax=axes[0]
)
axes[0].set_title("Sign-ups by Source")
axes[0].set_xlabel("Number of Sign-ups")
axes[0].set_ylabel("Source")

sns.barplot(
    data=region_counts, 
    x='Signups', 
    y='Region', 
    hue='Region', 
    dodge=False, 
    palette='pastel', 
    legend=False,
    ax=axes[1]
)
axes[1].set_title("Sign-ups by Region")
axes[1].set_xlabel("Number of Sign-ups")
axes[1].set_ylabel("")

sns.barplot(
    data=plan_counts, 
    x='Plan', 
    y='Signups', 
    hue='Plan', 
    dodge=False, 
    palette='pastel', 
    legend=False,
    ax=axes[2]
)
for i, v in enumerate(plan_counts['Signups']):
    axes[2].text(i, v + 2, str(v), ha='center', fontweight='bold')
axes[2].set_title("Sign-ups by Plan")
axes[2].set_xlabel("Plan Selected")
axes[2].set_ylabel("")

plt.tight_layout()
plt.show()

heatmap_data = signups_by_factors.pivot_table(
    index='source', 
    columns='region', 
    values='Signups', 
    aggfunc='sum', 
    fill_value=0
)

plt.figure(figsize=(10,6))
sns.heatmap(heatmap_data, annot=True, fmt='d', cmap='Blues')
plt.title("Sign-ups by Source and Region")
plt.ylabel("Source")
plt.xlabel("Region")
plt.tight_layout()
plt.show()

stacked_data = signups_by_factors.pivot_table(
    index='source', 
    columns='plan_selected', 
    values='Signups', 
    aggfunc='sum', 
    fill_value=0
)

stacked_data.plot(
    kind='bar', 
    stacked=True, 
    figsize=(10,6), 
    colormap='Pastel1'
)
plt.title("Sign-ups by Source, Stacked by Plan")
plt.xlabel("Source")
plt.ylabel("Number of Sign-ups")
plt.xticks(rotation=45)
plt.legend(title="Plan Selected")
plt.tight_layout()
plt.show()


#### Marketing opt-in counts by gender

In [ ]:
marketing_by_gender = (
    customers.groupby(['gender', 'marketing_opt_in'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

print("Marketing Opt-in by Gender:")
display(marketing_by_gender)

marketing_by_gender.plot(
    x='gender', kind='bar', stacked=True, figsize=(8,4),
    title='Marketing Opt-in Counts by Gender'
)
plt.ylabel('Count')
plt.show()


#### Age summary: min, max, mean, median, null count

In [ ]:
missing_ages = customers['age_source'] == 'Imputed'
unusual_count = customers['age_cleaned'].apply(lambda x: isinstance(x, str) and x.startswith("UNUSUAL")).sum()
numeric_age = customers['age_cleaned'].apply(lambda x: x if isinstance(x, int) else np.nan)

summary_stats = pd.DataFrame({
    'Min': [int(numeric_age.min())],
    'Max': [int(numeric_age.max())],
    'Mean': [round(numeric_age.mean(), 1)],
    'Median': [int(numeric_age.median())],
    'Missing Count': [sum(missing_ages)],
    'Unusual Count': [unusual_count]
})

display(summary_stats)

print("Missing Ages:")
display(customers.loc[missing_ages, ['customer_id','email', 'age', 'age_cleaned', 'age_source']])

unusual_ages = customers[customers['age_source'] == 'Unusual']
print("\nUnusual Ages:")
display(unusual_ages[['customer_id', 'age', 'age_cleaned', 'age_source']])

### Stretch Task

##### Load and Inspect Support Tickets

In [ ]:
tickets['customer_id'] = tickets['customer_id'].astype(str).str.strip()

tickets['ticket_date'] = pd.to_datetime(tickets['ticket_date'], errors='coerce')

print(f"Support tickets: {tickets.shape}")
tickets.head()

##### Merge Customers with Support Tickets

In [ ]:
customer_tickets = pd.merge(
    customers[['customer_id', 'signup_date', 'plan_selected', 'region']],
    tickets[['customer_id', 'ticket_date']],
    on='customer_id',
    how='inner' 
)
customer_tickets.head()

##### Count how many customers contacted support within 2 weeks of sign-up

In [ ]:
customer_tickets['days_since_signup'] = (customer_tickets['ticket_date'] - customer_tickets['signup_date']).dt.days

tickets_within_2weeks = customer_tickets[customer_tickets['days_since_signup'] <= 14]

num_customers_within_2weeks = tickets_within_2weeks['customer_id'].nunique()

print(f"Number of customers who contacted support within 2 weeks of signup: {num_customers_within_2weeks}")


##### Summarise support activity by plan and region (Group by plan and region)

In [ ]:
support_summary = (
    customer_tickets
    .groupby(['plan_selected', 'region'])
    .agg(
        total_tickets=('ticket_date', 'count'),
        tickets_within_2weeks=('days_since_signup', lambda x: (x <= 14).sum()),
        unique_customers=('customer_id', 'nunique')
    )
    .reset_index()
)

display(support_summary)

pivot_total = support_summary.pivot(index='plan_selected', columns='region', values='total_tickets').fillna(0)
pivot_2weeks = support_summary.pivot(index='plan_selected', columns='region', values='tickets_within_2weeks').fillna(0)

plt.figure(figsize=(14,6))

plt.subplot(1,2,1)
sns.heatmap(pivot_total, annot=True, fmt='.0f', cmap='Blues')
plt.title("Total Support Tickets by Plan and Region")
plt.ylabel("Plan Selected")
plt.xlabel("Region")

plt.subplot(1,2,2)
sns.heatmap(pivot_2weeks, annot=True, fmt='.0f', cmap='Greens')
plt.title("Tickets Within 2 Weeks by Plan and Region")
plt.ylabel("Plan Selected")
plt.xlabel("Region")

plt.tight_layout()
plt.show()

### Business Question Answers

#### 1. Which acquisition source brought in the most users last month?

In [ ]:
last_month_end = customers['signup_date'].max().replace(day=1) - pd.DateOffset(days=1)
last_month_start = last_month_end.replace(day=1)

recent_signups = customers[
    (customers['signup_date'] >= last_month_start) &
    (customers['signup_date'] <= last_month_end)
]

source_counts = recent_signups['source'].value_counts()
print(source_counts)

plt.figure(figsize=(8,5))
sns.barplot(
   x=source_counts.index,
    y=source_counts.values,
    palette="viridis",
    hue=source_counts.index,
    legend=False
)
plt.title("User Sign-Ups by Acquisition Source (Last Month)", fontsize=14, weight='bold')
plt.xlabel("Acquisition Source", fontsize=12)
plt.ylabel("Number of Sign-Ups", fontsize=12)
plt.xticks(rotation=30)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


#### 2. Which region shows signs of missing or incomplete data?

In [ ]:
customers['missing_fields'] = customers.isna().sum(axis=1)

region_missing_summary = (
    customers.groupby('region')['missing_fields']
    .mean()
    .sort_values(ascending=False)
)

print("Average Missing Fields per User by Region:")
display(region_missing_summary)

heatmap_data = region_missing_summary.to_frame(name='Average Missing Fields').T

plt.figure(figsize=(10, 2.5))
sns.heatmap(heatmap_data, annot=True, cmap='coolwarm', fmt=".2f", cbar_kws={'label': 'Average Missing Fields'})
plt.title("Average Missing Fields per User by Region", fontsize=14, weight='bold')
plt.xlabel("")
plt.ylabel("")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()



#### 3. Are older users more or less likely to opt in to marketing?

In [ ]:
marketing_by_age = customers[customers['age_source'] != 'Unusual'].copy()
marketing_by_age['age_group'] = pd.cut(marketing_by_age['age_cleaned'], bins=[16,25,35,45,55,65,100], labels=['16-25','26-35','36-45','46-55','56-65','66+'])

opt_in_counts = marketing_by_age.groupby('age_group', observed=False)['marketing_opt_in'].value_counts(normalize=True).unstack(fill_value=0)
display(opt_in_counts)

ax = opt_in_counts.plot(
    kind='bar',
    stacked=True,
    figsize=(8,5),
    color=['#66b3ff','#ff9999']
)

plt.title("Marketing Opt-In Rate by Age Group", fontsize=14, weight='bold')
plt.xlabel("Age Group", fontsize=12)
plt.ylabel("Proportion of Users", fontsize=12)
plt.legend(title="Marketing Opt-In", labels=['No','Yes'])
plt.xticks(rotation=0)
plt.ylim(0,1)
plt.tight_layout()
plt.show()


#### 4. Which plan is most commonly selected, and by which age group?

In [ ]:
plan_age = customers[customers['age_source'] != 'Unusual'].copy()
plan_age['age_group'] = pd.cut(plan_age['age_cleaned'], bins=[16,25,35,45,55,65,100], labels=['16-25','26-35','36-45','46-55','56-65','66+'])

plan_age_counts = plan_age.groupby(['plan_selected','age_group'], observed=False).size().unstack(fill_value=0)
display(plan_age_counts)

plan_age_counts.T.plot(
    kind='bar', 
    stacked=True, 
    figsize=(10,6), 
    colormap='Pastel1'
)

plt.title("Age Group Distribution by Plan", fontsize=14, weight='bold')
plt.xlabel("Age Group", fontsize=12)
plt.ylabel("Number of Users", fontsize=12)
plt.xticks(rotation=0)
plt.legend(title="Plan")
plt.tight_layout()
plt.show()


#### 5. Which plan’s users are most likely to contact support?

In [ ]:
support_by_plan = customer_tickets.groupby('plan_selected')['customer_id'].nunique()
support_by_plan = support_by_plan.sort_values(ascending=False)
display(support_by_plan)

plt.figure(figsize=(8,5))
sns.barplot(
    x=support_by_plan.index, 
    y=support_by_plan.values, 
    palette='pastel',
    hue=support_by_plan.index,
    legend=False
)
plt.title("Number of Customers Contacting Support by Plan", fontsize=14, weight='bold')
plt.xlabel("Plan Selected", fontsize=12)
plt.ylabel("Number of Customers", fontsize=12)
for i, v in enumerate(support_by_plan.values):
    plt.text(i, v + 0.2, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()
